# 📄 Gerando Dataset de Instruções para Fine‑Tuning a partir de Documentos

Este tutorial ensina como transformar conhecimento bruto — contido em manuais, guias ou artigos (PDF/Texto) — em um conjunto de dados no formato **instruction‑input‑output**, pronto para ser usado em fine‑tuning de modelos de linguagem (como o que fizemos com LoRA).

**Por que isso é importante?**
- Modelos pré‑treinados possuem grande capacidade, mas frequentemente carecem de conhecimento específico de domínio.
- Ao criar pares pergunta‑resposta baseados em documentos reais, podemos ensinar o modelo a responder corretamente sobre aquele domínio.
- O formato JSONL utilizado é o padrão para *instruction tuning*, amplamente adotado em projetos como Alpaca, Dolly, etc.

**O que você aprenderá:**
- Extrair texto de arquivos PDF.
- Dividir o texto em trechos (chunks) adequados para processamento.
- Usar tanto um modelo **seq2seq** (encoder‑decoder) quanto um modelo **causal** (autoregressivo) para gerar automaticamente triplas `instruction`, `input` (opcional) e `output`.
- Estruturar e salvar os dados em JSONL.
- Comparar a qualidade dos dados gerados com uma extração manual simples.

## 📚 1. Fundamentos da Geração de Dados Instrucionais

O objetivo é, dado um trecho de texto $D$ (ex.: um parágrafo de manual), produzir uma tripla $(I, X, O)$ onde:
- $I$: instrução (pergunta ou comando)
- $X$: entrada adicional (opcional, pode ser vazia)
- $O$: saída desejada, baseada no conteúdo de $D$

Formalmente, queremos modelar uma distribuição condicional:

$$P(I, X, O \mid D)$$

Utilizamos um LLM para amostrar dessas distribuições, fornecendo um *prompt* que instrui o modelo a gerar o JSON desejado a partir do contexto.

**Por que isso funciona?**  
Modelos de linguagem modernos, quando condicionados com prompts adequados, conseguem realizar *in‑context learning* – eles entendem a tarefa descrita e produzem texto estruturado. A qualidade depende fortemente:
- Da capacidade do modelo (ex.: 7B+ parâmetros).
- Da clareza do prompt.
- Da temperatura (controla a criatividade).

Neste notebook demonstraremos dois paradigmas:
- **Seq2Seq** (ex.: FLAN‑T5): recebe o texto inteiro e gera a saída condicionada ao *encoder*.
- **Causal** (ex.: GPT‑2): gera a continuação de um prompt, simulando a tarefa como um "completamento" de texto.

> **Nota didática:** Utilizaremos modelos pequenos para demonstrar o fluxo. Em aplicações reais, recomenda‑se modelos maiores (LLaMA 3, GPT‑4, etc.) para melhores resultados.

## 📦 2. Requisitos

Instale as bibliotecas necessárias (descomente a linha se ainda não as tiver):

In [1]:
!pip install transformers datasets pypdf2 accelerate sentencepiece

In [2]:
import json
import re
from pathlib import Path

from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

/home/henderson/Documentos/Rag/Pipeline-RAG-com-Fine-Tuning-LoRA-e-Disponibiliza-o-via-API-RESTful/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 📥 3. Carregar e Extrair Texto de um PDF

Vamos usar um arquivo PDF de exemplo: `manual.pdf`. O código abaixo extrai todo o texto de todas as páginas.

In [3]:
def extract_text_from_pdf(pdf_path):
    """Extrai texto de um arquivo PDF."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

# Substitua pelo caminho do seu PDF
pdf_path = "RelatGestSaud.pdf"
full_text = extract_text_from_pdf(pdf_path)
print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

Total de caracteres extraídos: 647679

--- INÍCIO DO TEXTO ---

RELATÓRIO DE GESTÃO

MENSAGEM DO MINISTRO 2
VISÃO GERAL ORGANIZACIONAL  
E GOVERNANÇA 3
1.1  Identificação da UPC  
 (Unidade Prestadora de Contas) 4
1.2    Estrutura Organizacional 5
1.3    Cadeia de Valor 9
1.4   Mapa Estratégico 10 
1.5    Políticas Estratégicas 11 
1.6    Planejamento e Monitoramento 13 
1.7    Descrição dos objetivos do Exercício 17 
1.8    Monitoramento dos Instrumentos  
              de Planejamento 19
1.9    Estrutura de Governança  23
1.10  Oportunidades e Perspectivas


## ✂️ 4. Dividir o Texto em *Chunks* (Pedaços)

Modelos de linguagem têm um limite máximo de tokens de entrada (janela de contexto). Precisamos quebrar o texto em segmentos que caibam nessa janela, mas mantendo significado.  
Estratégias comuns:
- Divisão por parágrafos.
- Divisão com sobreposição (*sliding window*) para evitar perda de contexto nas bordas.

Aqui faremos uma divisão simples: a cada `max_chunk_chars` caracteres, garantindo que a quebra ocorra em um espaço (para não cortar palavras).

In [4]:
def chunk_text(text, max_chunk_chars=800):
    """Divide o texto em blocos de no máximo max_chunk_chars caracteres."""
    words = text.split()
    chunks = []
    current_chunk = ""
    for word in words:
        if len(current_chunk) + len(word) + 1 <= max_chunk_chars:
            current_chunk += (" " if current_chunk else "") + word
        else:
            chunks.append(current_chunk)
            current_chunk = word
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

chunks = chunk_text(full_text)  # pequeno para demonstração
print(f"Número de chunks: {len(chunks)}")
print(f"Exemplo de chunk (primeiro):\n{chunks[0][:300]}...")

Número de chunks: 799
Exemplo de chunk (primeiro):
RELATÓRIO DE GESTÃO MENSAGEM DO MINISTRO 2 VISÃO GERAL ORGANIZACIONAL E GOVERNANÇA 3 1.1 Identificação da UPC (Unidade Prestadora de Contas) 4 1.2 Estrutura Organizacional 5 1.3 Cadeia de Valor 9 1.4 Mapa Estratégico 10 1.5 Políticas Estratégicas 11 1.6 Planejamento e Monitoramento 13 1.7 Descrição ...


## 🎯 5. Projetar o *Prompt* para Geração

O prompt deve instruir o modelo a produzir um JSON válido com os campos `instruction`, `input` e `output`. Como trabalharemos com duas arquiteturas, precisamos de prompts ligeiramente diferentes:

- **Seq2Seq**: o prompt é enviado diretamente como entrada, e o modelo gera a saída condicionada. Exemplo:
```
You are an AI assistant... 
Text: {chunk}
JSON: 
```
- **Causal**: o modelo vê o prompt como um prefixo e deve continuar a sequência. Precisamos estruturar o prompt de forma que o modelo "complete" com o JSON. Exemplo (estilo Alpaca):
```
### Instruction: ... 
### Input: {chunk}
### Response: 
```

Utilizaremos `temperature=0.3` para equilibrar criatividade e fidelidade.

## 🤖 6. Carregar Modelos de Linguagem: Seq2Seq e Causal

Carregaremos dois modelos:
- `google/flan-t5-base` (seq2seq) – pipeline `text2text-generation`
- `gpt2` (causal) – pipeline `text-generation`

> **Em produção:** Substitua por modelos maiores como `mistralai/Mistral-7B-Instruct` ou `meta-llama/Llama-3-8B-Instruct`.

In [5]:
causal_id = "microsoft/Phi-3-mini-4k-instruct"
causal_tokenizer = AutoTokenizer.from_pretrained(causal_id)

# Adicione device_map="auto" aqui
causal_model = AutoModelForCausalLM.from_pretrained(causal_id, device_map="auto")

generator_causal = pipeline(
    "text-generation",
    model=causal_model,
    tokenizer=causal_tokenizer,
    device_map="auto" # Adicione aqui também
)

Loading weights: 100%|██████████| 195/195 [00:00<00:00, 9796.14it/s]


## 🔄 7. Gerar Triplas com o Modelo Seq2Seq

Iteramos sobre todos os chunks, montamos o prompt e extraímos a tripla usando um extrator robusto. Apenas chunks com conteúdo significativo são processados.

In [6]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def is_bad_chunk(text, min_len=80):
    text = text.strip()
    if len(text) < min_len:
        return True
    words = text.split()
    if len(set(words)) < 10:
        return True
    strange_ratio = sum(1 for c in text if not c.isalnum() and c not in " .,;:-()[]{}") / max(len(text), 1)
    if strange_ratio > 0.3:
        return True
    return False

def run_model(generator, prompt):
    output = generator(
        prompt,
        max_new_tokens=800,   # <--- AUMENTE ISSO PARA 800!
        temperature=0.3,      
        top_p=0.9,
        do_sample=True,       
        return_full_text=False
    )
    if isinstance(output, list):
        return output[0]["generated_text"]
    return str(output)

def extract_triple(text, chunk):
    try:
        # Usa Expressão Regular (Regex) para encontrar apenas o que está entre { e }
        match = re.search(r'\{.*\}', text, re.DOTALL)
        
        if match:
            json_str = match.group(0)
            data = json.loads(json_str)
            
            # Padroniza as chaves para minúsculo para evitar erros
            data = {k.lower(): v for k, v in data.items()}
            
            return {
                "instruction": data.get("instruction", ""),
                "input": data.get("input", chunk),
                "output": data.get("output", "")
            }
        else:
            return None
            
    except json.JSONDecodeError:
        return None

# =========================================================
# PREPARA CHUNKS LIMPOS
# =========================================================
clean_chunks = [clean_text(c) for c in chunks if not is_bad_chunk(c)]
print(f"Chunks válidos: {len(clean_chunks)}")


Chunks válidos: 799


## 🔄 8. Gerar Triplas com Modelo Causal (Autoregressivo)

Agora demonstramos o mesmo processo, mas usando um modelo causal (GPT‑2). Aqui o prompt deve ser construído para que o modelo "complete" com o JSON. Usaremos um formato inspirado no Alpaca.

In [7]:
PROMPT_CAUSAL = """<|system|>
Você é um especialista em inteligência artificial e gerador de datasets de alta qualidade para fine-tuning.
Sua tarefa é ler um bloco de texto (Contexto) e transformá-lo em EXATAMENTE UM bloco JSON válido, sem qualquer texto adicional, explicações ou blocos de código markdown.

Formato do JSON estritamente obrigatório:
{
  "instruction": "Uma pergunta ou comando direto gerado por você com base estrita no contexto.",
  "input": "O trecho exato ou contexto resumido que foi usado para responder à pergunta.",
  "output": "A resposta exata, factual, detalhada e precisa para a instrução criada."
}

REGRAS CRÍTICAS:
1. Retorne APENAS o objeto JSON puro. Não use crases e não escreva nada antes ou depois.
2. A "instruction" deve ser uma pergunta inteligente que alguém faria sobre o texto.
3. Não invente informações além do que está explicitamente escrito no contexto.

Exemplo de Saída Esperada:
{
  "instruction": "Qual é o principal objetivo da Sala de Apoio à Gestão Estratégica (SAGE)?",
  "input": "A Sala de Apoio à Gestão Estratégica (SAGE) disponibiliza informações em saúde para apoiar a tomada de decisão.",
  "output": "O principal objetivo da SAGE é disponibilizar dados e informações estratégicas de saúde para subsidiar e apoiar os gestores na tomada de decisão."
}<|end|>
<|user|>
Gere o JSON para o seguinte contexto:

Contexto:
{chunk}

JSON:<|end|>
<|assistant|>"""

triplas_causal = []
# Mude para testar os primeiros 30 chunks e ver se saem diferentes
# TESTE DE DIAGNÓSTICO: Apenas 1 chunk!
amostra_chunks = chunks[:115] 

for i, chunk in enumerate(amostra_chunks):
    texto_do_chunk = chunk.strip()
    if not texto_do_chunk:
        continue
        
    prompt = PROMPT_CAUSAL.replace("{chunk}", texto_do_chunk)
    
    try:
        result = run_model(generator_causal, prompt)
        
        if result.startswith(prompt):
            generated = result[len(prompt):].strip()
        else:
            generated = result
        triple = extract_triple(generated, texto_do_chunk)

        if triple is not None and len(triple["output"]) > 10 and len(triple["instruction"]) > 5:
            triplas_causal.append(triple)
        
        if i % 10 == 0:
            print(f"⏳ Processando chunk {i}/{len(amostra_chunks)}... (Triplas válidas capturadas até agora: {len(triplas_causal)})", flush=True)

    except Exception as e:
        print(f"Erro no chunk {i}: {e}")
print(f"Triplas geradas (causal): {len(triplas_causal)}")

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


⏳ Processando chunk 0/115... (Triplas válidas capturadas até agora: 1)


[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

⏳ Processando chunk 10/115... (Triplas válidas capturadas até agora: 11)


[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

...

⏳ Processando chunk 100/115... (Triplas válidas capturadas até agora: 100)


[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

⏳ Processando chunk 110/115... (Triplas válidas capturadas até agora: 110)


[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Triplas geradas (causal): 114


## 🧹 9. Pós‑processamento e Limpeza

Removemos espaços extras, tratamos `input` inválidos e filtramos triplas com output muito curto.

In [8]:
def clean_triple(triple):
    for key in triple:
        triple[key] = triple[key].strip()
    if triple["input"].lower() in ("none", "null", "n/a", ""):
        triple["input"] = ""
    if len(triple["output"]) < 20 or len(triple["instruction"]) < 5:
        return None
    return triple

def limpar_lista(triplas):
    return [t for t in (clean_triple(x) for x in triplas) if t is not None]

triplas_causal = limpar_lista(triplas_causal)

print(f"Triplas causal após limpeza: {len(triplas_causal)}")

Triplas causal após limpeza: 114


## 💾 10. Salvar os Datasets em JSONL

Cada linha será um objeto JSON independente, pronto para uso com `load_dataset`.

In [9]:
def salvar_jsonl(triplas, nome_arquivo):
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        for t in triplas:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"Dataset salvo em {nome_arquivo}")

salvar_jsonl(triplas_causal, "dataset_causal.jsonl")

Dataset salvo em dataset_causal.jsonl


## 🔬 11. Comparação: Modelo Seq2Seq vs. Causal

Exibimos alguns exemplos gerados por cada modelo para comparar estilos e qualidade.

In [10]:
def mostrar_exemplo(triplas, titulo, n=3):
    print(f"\n=== {titulo} ===")
    for i, t in enumerate(triplas[:n], 1):
        print(f"\n--- Exemplo {i} ---")
        print(json.dumps(t, indent=2, ensure_ascii=False))

mostrar_exemplo(triplas_causal, "Causal (GPT-2)")



=== Causal (GPT-2) ===

--- Exemplo 1 ---
{
  "instruction": "Qual é a principal responsabilidade da Unidade Prestadora de Contas (UPC) no contexto do Relatório de Gestão Mensal do Ministro 2?",
  "input": "RELATÓRIO DE GESTÃO MENSAGEM DO MINISTRO 2 VISÃO GERAL ORGANIZACIONAL E GOVERNANÇA 1.1 Identificação da UPC (Unidade Prestadora de Contas) 4 1.2 Estrutura Organizacional 5 1.3 Cadeia de Valor 9 1.4 Mapa Estratégico 10 1.5 Políticas Estratégicas 11 1.6 Planejamento e Monitoramento 13 1.7 Descrição dos objetivos do Exercício 17 1.8 Monitoramento dos Instrumentos de Planejamento 19 1.9 Estrutura de Governança 23 1.10 Oportunidades e Perspectivas 26 CONFORMIDADE E EFICIÊNCIA DA GESTÃO 203 3.1 Gestão Orçamentária e Financeira 204 3.2 Gestão de Pessoas 217 3.3 Gestão de Licitações e Contratos 226 3.4 Gestão Patrimonial e Infraestrutura 234 3.5 Gestão de Tecnologia da Informação 236 3.6 Gestão de Custos 244 3.7 Sustentabilidade Ambiental 245 3.8 Relacionamento com a Sociedade 247RESUL TAD

## ❓ 12. Exemplo Prático: Pergunta e Resposta com Seq2Seq e Causal

Para ilustrar como os dois paradigmas podem ser usados diretamente como sistemas de QA (Question Answering), vamos tomar um trecho concreto do manual e fazer uma pergunta. O modelo receberá o contexto e a pergunta, e deverá gerar uma resposta.

Escolhemos o primeiro chunk (limpo), que contém a informação da capacidade do refrigerador.

In [11]:
# Seleciona um chunk de exemplo (o primeiro chunk limpo)
contexto = clean_chunks[0]
pergunta = "Como é feito o monitoramento dos instrumentos de planejamento e a gestão orçamentária dentro da estrutura de governança?"

print(f"Contexto:\n{contexto}\n")
print(f"Pergunta: {pergunta}\n")

# ---------- CAUSAL ----------
prompt_causal_qa = f"""
Você é um assistente que responde perguntas usando apenas o contexto fornecido.

Contexto:
{contexto}

Pergunta:
{pergunta}

Resposta:
"""

resposta_causal = run_model(generator_causal, prompt_causal_qa)
# Remove possível repetição do prompt
if resposta_causal.startswith(prompt_causal_qa):
    resposta_causal = resposta_causal[len(prompt_causal_qa):].strip()
print(f"\n--- Resposta (Causal - Phi-3) ---")
print(resposta_causal.strip())

print("\nNota: O modelo seq2seq foi mais conciso e extraiu exatamente a informação relevante. O modelo causal adicionou um pouco mais de texto e até repetiu em inglês, mostrando a necessidade de um bom controle de geração (temperatura, max tokens, stopping criteria). Ambos, porém, conseguiram capturar o valor correto presente no contexto.")

[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Contexto:
RELATÓRIO DE GESTÃO MENSAGEM DO MINISTRO 2 VISÃO GERAL ORGANIZACIONAL E GOVERNANÇA 3 1.1 Identificação da UPC (Unidade Prestadora de Contas) 4 1.2 Estrutura Organizacional 5 1.3 Cadeia de Valor 9 1.4 Mapa Estratégico 10 1.5 Políticas Estratégicas 11 1.6 Planejamento e Monitoramento 13 1.7 Descrição dos objetivos do Exercício 17 1.8 Monitoramento dos Instrumentos de Planejamento 19 1.9 Estrutura de Governança 23 1.10 Oportunidades e Perspectivas 26 CONFORMIDADE E EFICIÊNCIA DA GESTÃO 203 3.1 Gestão Orçamentária e Financeira 204 3.2 Gestão de Pessoas 217 3.3 Gestão de Licitações e Contratos 226 3.4 Gestão Patrimonial e Infraestrutura 234 3.5 Gestão de Tecnologia da Informação 236 3.6 Gestão de Custos 244 3.7 Sustentabilidade Ambiental 245 3.8 Relacionamento com a Sociedade 247RESUL TADOS E

Pergunta: Como é feito o monitoramento dos instrumentos de planejamento e a gestão orçamentária dentro da estrutura de governança?


--- Resposta (Causal - Phi-3) ---
O monitoramento dos ins

## 🎓 13. Conclusão e Próximos Passos

Você agora sabe como transformar documentos em datasets de instruções utilizando **dois paradigmas** de modelos de linguagem. Este dataset pode alimentar o fine‑tuning que aprendemos no notebook anterior, fechando o ciclo completo: **documento → dataset → modelo especializado**.

**Resumo do fluxo:**
1. **Extração** de texto do PDF.
2. **Divisão** em chunks compatíveis com o modelo.
3. **Geração** via LLM (seq2seq ou causal) com prompt estruturado.
4. **Limpeza** e validação do JSON.
5. **Exportação** para JSONL.

**Possíveis melhorias:**
- Usar *few‑shot examples* no prompt para guiar o estilo.
- Aplicar *self‑consistency* (gerar várias respostas e escolher a melhor).
- Validar a fidelidade da resposta em relação ao texto original (usando similaridade de embeddings).
- Substituir os modelos base por versões mais potentes (ex.: Llama 3 8B com 4‑bit quantização).

Agora você está pronto para criar seus próprios dados de treinamento e levar seus modelos ao próximo nível!